In [1]:
import pandas as pd
import sys
sys.path.insert(0, "..")
from utils import fill_empty_rows, import_datasets_to_db, create_file_path
from sqlalchemy import create_engine

In [2]:
# Read the .csv file
df_toyota = pd.read_csv(create_file_path("toyotaR.csv"))

In [3]:
# Check the df info
df_toyota.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5059 entries, 0 to 5058
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Style            3591 non-null   object
 1   HDW              3569 non-null   object
 2   Toyota P/N       3590 non-null   object
 3   Other P/N        2081 non-null   object
 4   Tag              1719 non-null   object
 5   Stkr ENG, Trans  424 non-null    object
 6   Year             3082 non-null   object
 7   Model            3428 non-null   object
 8   Model Code(s)    3400 non-null   object
 9   Description      1511 non-null   object
 10  SecType          3245 non-null   object
 11  Calibrations     464 non-null    object
 12  Example Vin      1135 non-null   object
 13  Prog             414 non-null    object
dtypes: object(14)
memory usage: 553.5+ KB


### Columns to keep
- Style
- hardware
- Toyota P/N
- Other P/N

In [8]:
# Mapping all cols will keep
cols_to_keep = ["Style", "HDW", "Toyota P/N", "Other P/N"]

In [9]:
# Create a df with needed cols
df_toyota_keep_cols = df_toyota[cols_to_keep]

In [10]:
# Remove rows where all cols have null (NaN) values,
# keeping only rows with at least one value (non-null) entry.
df_toyota_wo_all_colsNull = df_toyota_keep_cols[~df_toyota_keep_cols.isnull().all(axis=1)]

In [11]:
# Keep only rows where the cols have > 1 non-null values.
df_toyota_rm_all_nulls = df_toyota_wo_all_colsNull[df_toyota_wo_all_colsNull.notnull().sum(axis=1) != 1]

In [12]:
# Index list of rows filled out with the columns
index_list_repeated_cols = df_toyota_rm_all_nulls[df_toyota_rm_all_nulls['Style'] == 'Style'].index

In [13]:
# Drop rows with col names
df_toyota_dropped_cols = df_toyota_rm_all_nulls.drop(index=index_list_repeated_cols)

In [14]:
# Dict to rename the columns
rn_cols_dict = {
    "Style": "pn1",
    "HDW": "hardware",
    "Toyota P/N": "pn2",
    "Other P/N": "pn3"
}

In [ ]:
# Rename columns
df_toyota_rn_cols = df_toyota_dropped_cols.rename(columns=rn_cols_dict)

In [16]:
# Reorder hardware column
df_toyota_reorder_cols = df_toyota_rn_cols[["hardware", "pn1", "pn2", "pn3"]]  

In [17]:
# Call the function to fill out null values
df_toyota_fill_nulls = fill_empty_rows(df_toyota_reorder_cols)

In [18]:
# Check df info after changes
df_toyota_fill_nulls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3569 entries, 0 to 5058
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   hardware  3569 non-null   object
 1   pn1       3569 non-null   object
 2   pn2       3569 non-null   object
 3   pn3       3569 non-null   object
dtypes: object(4)
memory usage: 139.4+ KB


In [19]:
# Call the function to import the df cleaned to the postgreSQLdb
import_datasets_to_db(df_toyota_fill_nulls, 'part_numbers_toyota')

569